In [0]:
                                                    #AUTOMIND BATCH 1
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch1/driver_master.csv")
)

df.write.mode("overwrite").saveAsTable("workspace.intake.driver_master")
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch1/route_master.csv")
)

df.write.mode("overwrite").saveAsTable("workspace.intake.route_master")
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch1/vehicle_master.csv")
)


df.write.mode("overwrite").saveAsTable("workspace.intake.vehicle_master")
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch1/trip_master.csv")
)


df.write.mode("overwrite").saveAsTable("workspace.intake.trip_master")

                                                    #AUTOMIND BATCH 2
df = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/intake/source_system/AutoMind_Batch2/driver_behavior_001.json")
)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.driver_behavior")

df = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/intake/source_system/AutoMind_Batch2/engine_001.json")
)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.core_engine")

df = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/intake/source_system/AutoMind_Batch2/gps_001.json")
)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.gps")

df = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/intake/source_system/AutoMind_Batch2/telemetry_001.json")
)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.core_telemetry")




                                                    #AUTOMIND BATCH 3
import pandas as pd
pdf = pd.read_excel(
    "/Volumes/workspace/intake/source_system/AutoMind_Batch3/fuel_transactions.xlsx"
)
df = spark.createDataFrame(pdf)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.fuel_transactions")

df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch3/insurance_claims.csv")
)
df.write.mode("overwrite").saveAsTable("workspace.intake.insurance_claims")

df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/intake/source_system/AutoMind_Batch3/maintenance.csv")
)
df.write.mode("overwrite").saveAsTable("workspace.intake.maintenance")
df = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/intake/source_system/AutoMind_Batch3/weather_001.json")
)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.intake.weather")



In [0]:
                                                        #AUTOMIND BATCH 4 INGESTION

from pyspark.sql.functions import current_timestamp, lit, col
import pandas as pd

# READ ACCIDENT REPORT PDFs

accident_df = (
    spark.read
        .format("binaryFile")
        .option("pathGlobFilter", "*.pdf")
        .load("/Volumes/workspace/intake/source_system/AutoMind_Batch4/ProfessionalSyntheticInsuranceDataset_v2/AccidentReports/")
)

# READ INSURANCE CLAIM PDFs

claims_df = (
    spark.read
        .format("binaryFile")
        .option("pathGlobFilter", "*.pdf")
        .load("/Volumes/workspace/intake/source_system/AutoMind_Batch4/ProfessionalSyntheticInsuranceDataset_v2/InsuranceClaims/")
)

# ADD INGESTION METADATA

intake_accident = (
    accident_df
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("file_name", col("_metadata.file_name"))
        .withColumn("file_size", col("_metadata.file_size"))
        .withColumn("document_type", lit("Accident Report"))
        .withColumn("batch_id", lit("Batch4"))
        .withColumn("ingestion_timestamp", current_timestamp())
)

intake_claim = (
    claims_df
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("file_name", col("_metadata.file_name"))
        .withColumn("file_size", col("_metadata.file_size"))
        .withColumn("document_type", lit("Insurance Claim"))
        .withColumn("batch_id", lit("Batch4"))
        .withColumn("ingestion_timestamp", current_timestamp())
)

# WRITE PDF TABLES TO INTAKE (BRONZE)

intake_accident.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.intake.accident_reports_batch4")

intake_claim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.intake.insurance_claims_batch4")

# READ METADATA EXCEL

metadata_pd = pd.read_excel(
    "/Volumes/workspace/intake/source_system/AutoMind_Batch4/ProfessionalSyntheticInsuranceDataset_v2/metadata.xlsx"
)

metadata_df = spark.createDataFrame(metadata_pd)

# WRITE METADATA TO INTAKE (BRONZE)


metadata_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.intake.metadata_batch4_files")